# 🧠 CorticalVAE: Bio-Inspired STRF Frontend for Auditory MemoryA convolutional VAE with a **learnable Spectro-Temporal Receptive Field (STRF)** frontend,inspired by [diffAudNeuro](https://github.com/pirl-lab/diffAudNeuro) (Lostanlen et al.).**Paradigm:** Agus et al. (2010) — rapid formation of auditory memories from noise.**Key design choices:**- Bio-inspired STRF conv layer (5 scales × 8 rates = 40 kernels) initialized from auditory neuroscience- Fully convolutional encoder/decoder- Online learning: single-sample SGD, no replay buffer- Input: cochlear spectrogram (128 channels, 1 ms resolution) from the NSL toolbox

## 1. Setup & Dependencies

In [ ]:
# ── Clone repo & install dependencies ──
import os, subprocess, sys, shutil

REPO_URL = "https://github.com/MeysamAmirsardari/Cortical_Encoder"
REPO_DIR = "Cortical_Encoder"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

# Copy needed modules to working directory
for item in ["nsl_toolbox", "config.py", "generate_stimuli.py"]:
    src = os.path.join(REPO_DIR, item)
    dst = item
    if os.path.isdir(src):
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile", "pydub"], check=True)
print("Setup complete.")

## 2. Configuration

In [ ]:
from dataclasses import dataclass, field
import numpy as np

@dataclass
class ExperimentConfig:
    # ── Audio / Stimulus ──
    generation_sr: int = 16000
    dur_half: float = 0.5
    dur_full: float = 1.0
    inter_trial_gap: float = 1.5
    n_noise: int = 100
    n_rn: int = 50
    n_refrn: int = 50
    frozen_seed: int = 42

    # ── Cochlear model (NSL) ──
    frmlen: float = 1       # ms per frame
    tc: float = 8           # time constant ms
    fac: float = -2         # linear
    octave_shift: float = 0 # 16 kHz

    # ── STRF frontend ──
    n_scales: int = 5
    n_rates: int = 8        # 4 abs rates × 2 signs
    scales: tuple = (0.25, 0.5, 1.0, 2.0, 4.0)          # cyc/oct
    rates: tuple = (-32, -16, -8, -4, 4, 8, 16, 32)      # Hz
    strf_kernel_t: int = 21  # temporal extent (frames = ms at 1ms)
    strf_kernel_f: int = 21  # spectral extent (channels)

    # ── VAE ──
    window_frames: int = 64  # input window: 64 ms
    n_freq: int = 128        # cochlear channels
    latent_dim: int = 64
    lr: float = 1e-4
    beta: float = 1.0        # KL weight

    # ── Derived ──
    @property
    def wav2aud_params(self):
        return [self.frmlen, self.tc, self.fac, self.octave_shift]

    @property
    def n_strf(self):
        return self.n_scales * self.n_rates  # 40

cfg = ExperimentConfig()
print(f"STRF kernels: {cfg.n_strf} ({cfg.n_scales} scales × {cfg.n_rates} rates)")
print(f"Input shape: (1, 1, {cfg.window_frames}, {cfg.n_freq})")
print(f"Latent dim: {cfg.latent_dim}")

## 3. Generate Stimuli & Compute Cochlear SpectrogramsWe generate the full Agus et al. (2010) block inline, compute cochlear spectrogramsvia `wav2aud`, and build a continuous stream for online learning.

In [ ]:
import numpy as np
from generate_stimuli import generate_block
from nsl_toolbox import wav2aud

print("Generating stimuli...")
trials, frozen_half, trial_order = generate_block()
n_trials = len(trials)
print(f"Trials: {n_trials} (N={np.sum(trial_order==0)}, RN={np.sum(trial_order==1)}, RefRN={np.sum(trial_order==2)})")

# ── Compute cochlear spectrograms per trial ──
print("Computing cochlear spectrograms (this takes a few minutes)...")
cochlear_trials = []
for i, trial in enumerate(trials):
    aud = wav2aud(trial["audio"], cfg.wav2aud_params)  # (1000, 128)
    cochlear_trials.append(aud)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{n_trials} trials done")

# ── Build continuous stream ──
cochlear_stream = np.concatenate(cochlear_trials, axis=0)  # (200000, 128)
print(f"Continuous cochlear stream: {cochlear_stream.shape}")

# ── Min-max normalize to [0, 1] ──
cmin, cmax = cochlear_stream.min(), cochlear_stream.max()
cochlear_stream = (cochlear_stream - cmin) / (cmax - cmin + 1e-12)
cochlear_stream = cochlear_stream.astype(np.float32)

# ── Build per-frame metadata ──
frames_per_trial = 1000  # 1s at 1ms resolution
frame_trial_idx = np.repeat(np.arange(n_trials), frames_per_trial)
frame_trial_type = np.repeat(trial_order, frames_per_trial)

# ── Track RefRN repetition onset frames (2nd half = frame 500 within trial) ──
refrn_indices = np.where(trial_order == 2)[0]
refrn_rep_onsets = refrn_indices * frames_per_trial + 500  # global frame indices

# ── Track RN repetition onset frames ──
rn_indices = np.where(trial_order == 1)[0]
rn_rep_onsets = rn_indices * frames_per_trial + 500

print(f"RefRN trials: {len(refrn_indices)}, RN trials: {len(rn_indices)}")
print(f"Total frames: {len(cochlear_stream)}, Windows per trial: {frames_per_trial - cfg.window_frames + 1}")

## 4. STRF Kernel InitializationWe generate biologically-inspired STRF kernels as outer products of temporal(damped sinusoid) and spectral (Gaussian 2nd derivative) components, followingthe NSL cortical model (Chi, Ru & Shamma, 2005).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def make_strf_kernels(cfg):
    """Generate STRF initialization kernels.

    Returns: Tensor of shape (n_strf, 1, kernel_t, kernel_f)
    """
    kernels = []
    kt, kf = cfg.strf_kernel_t, cfg.strf_kernel_f
    mid_t, mid_f = kt // 2, kf // 2

    for scale in cfg.scales:
        # Spectral component: Gaussian 2nd derivative
        f = np.arange(kf, dtype=np.float64) - mid_f
        # Normalize by scale
        sf = f * scale / 3.0  # /3 to keep kernel within bounds
        spec = (1 - sf**2) * np.exp(-sf**2 / 2.0)
        spec = spec / (np.abs(spec).max() + 1e-12)

        for rate in cfg.rates:
            # Temporal component: damped sinusoid
            t = np.arange(kt, dtype=np.float64) - mid_t
            abs_rate = abs(rate)
            # Temporal filter: windowed sinusoid
            env = np.exp(-3.5 * np.abs(t) / (abs_rate / 4.0 + 1e-6))
            if rate > 0:
                temp = env * np.cos(2 * np.pi * abs_rate / 1000.0 * t)
            else:
                temp = env * np.sin(2 * np.pi * abs_rate / 1000.0 * t)
            temp = temp / (np.abs(temp).max() + 1e-12)

            # Outer product → 2D STRF kernel
            kernel = np.outer(temp, spec)  # (kt, kf)
            kernels.append(kernel)

    kernels = np.stack(kernels, axis=0)[:, None, :, :]  # (40, 1, kt, kf)
    return torch.tensor(kernels, dtype=torch.float32)

strf_init = make_strf_kernels(cfg)
print(f"STRF init kernels: {strf_init.shape}")
print(f"  Scales: {cfg.scales}")
print(f"  Rates: {cfg.rates}")

## 5. CorticalVAE ModelArchitecture:- **STRF Frontend**: `Conv2d(1, 40, 21×21)` initialized with bio-inspired kernels (learnable)- **Encoder**: 4 strided conv layers (40→64→128→256→256) → flatten → FC → (μ, log σ²)- **Decoder**: FC → reshape → 4 transposed conv layers → sigmoid- **Loss**: BCE reconstruction + β·KL divergence

In [ ]:
class CorticalVAE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        nf = cfg.n_freq       # 128
        wf = cfg.window_frames # 64
        n_strf = cfg.n_strf    # 40
        kt = cfg.strf_kernel_t # 21
        kf = cfg.strf_kernel_f # 21
        z_dim = cfg.latent_dim # 64

        # ── STRF bio-inspired frontend (learnable) ──
        self.strf_conv = nn.Conv2d(1, n_strf, kernel_size=(kt, kf),
                                    padding=(kt//2, kf//2), bias=False)

        # ── Encoder: conv stack ──
        # After STRF: (B, 40, 64, 128)
        self.enc = nn.Sequential(
            nn.Conv2d(n_strf, 64, 4, stride=2, padding=1),  # → (64, 32, 64)
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 4, stride=2, padding=1),     # → (128, 16, 32)
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 4, stride=2, padding=1),    # → (256, 8, 16)
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 4, stride=2, padding=1),    # → (256, 4, 8)
            nn.BatchNorm2d(256), nn.ReLU(True),
        )
        # Flatten: 256 * 4 * 8 = 8192
        self.fc_mu     = nn.Linear(8192, z_dim)
        self.fc_logvar = nn.Linear(8192, z_dim)

        # ── Decoder ──
        self.fc_dec = nn.Linear(z_dim, 8192)
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(256, 256, 4, stride=2, padding=1),  # → (256, 8, 16)
            nn.BatchNorm2d(256), nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),  # → (128, 16, 32)
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),   # → (64, 32, 64)
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, stride=2, padding=1),     # → (1, 64, 128)
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.strf_conv(x)             # (B,40,64,128)
        h = self.enc(h)                    # (B,256,4,8)
        h = h.view(h.size(0), -1)         # (B,8192)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        h = self.fc_dec(z).view(-1, 256, 4, 8)
        return self.dec(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar, z

def vae_loss(recon, x, mu, logvar, beta=1.0):
    bce = F.binary_cross_entropy(recon, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return bce + beta * kl, bce, kl

# ── Instantiate ──
model = CorticalVAE(cfg).to(device)

# Initialize STRF conv with bio-inspired kernels
with torch.no_grad():
    model.strf_conv.weight.copy_(strf_init.to(device))

n_params = sum(p.numel() for p in model.parameters())
print(f"CorticalVAE: {n_params:,} parameters")
print(f"  STRF frontend: {cfg.n_strf} learnable kernels ({cfg.strf_kernel_t}×{cfg.strf_kernel_f})")
print(f"  Latent dim: {cfg.latent_dim}")

## 6. Online TrainingSingle-sample SGD: slide a 64-frame window across each trial (stride=1),one gradient update per window. The model learns and infers simultaneously —no batching, no replay.

In [ ]:
import time

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
model.train()

W = cfg.window_frames  # 64
stride = 1
windows_per_trial = frames_per_trial - W + 1  # 937

# ── Storage ──
all_losses = []           # per-step BCE
trial_mean_losses = []    # per-trial mean BCE
trial_types_list = []     # per-trial type label
latent_at_refrn_rep = []  # latent vectors at RefRN repetition onsets
latent_at_rn_rep = []     # latent vectors at RN repetition onsets

# Convert stream to tensor
stream_tensor = torch.tensor(cochlear_stream, dtype=torch.float32).to(device)

print(f"Online training: {n_trials} trials × {windows_per_trial} windows = {n_trials * windows_per_trial:,} steps")
t0 = time.time()

for trial_idx in range(n_trials):
    trial_start = trial_idx * frames_per_trial
    trial_losses = []
    trial_type = trial_order[trial_idx]

    for w in range(windows_per_trial):
        global_frame = trial_start + w
        x = stream_tensor[global_frame:global_frame + W, :]  # (64, 128)
        x = x.unsqueeze(0).unsqueeze(0)  # (1, 1, 64, 128)

        recon, mu, logvar, z = model(x)
        loss, bce, kl = vae_loss(recon, x, mu, logvar, beta=cfg.beta)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        bce_val = bce.item()
        trial_losses.append(bce_val)
        all_losses.append(bce_val)

        # Capture latent at repetition onsets
        if global_frame + W // 2 in refrn_rep_onsets:
            latent_at_refrn_rep.append(z.detach().cpu().numpy().flatten())
        if global_frame + W // 2 in rn_rep_onsets:
            latent_at_rn_rep.append(z.detach().cpu().numpy().flatten())

    trial_mean_losses.append(np.mean(trial_losses))
    trial_types_list.append(trial_type)

    if (trial_idx + 1) % 20 == 0:
        elapsed = time.time() - t0
        rate = (trial_idx + 1) / elapsed
        eta = (n_trials - trial_idx - 1) / rate
        print(f"  Trial {trial_idx+1:3d}/{n_trials} | "
              f"Mean BCE: {trial_mean_losses[-1]:.1f} | "
              f"ETA: {eta:.0f}s")

elapsed = time.time() - t0
print(f"\nTraining complete in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"Total gradient steps: {len(all_losses):,}")

trial_mean_losses = np.array(trial_mean_losses)
trial_types_list = np.array(trial_types_list)

## 7. Results### 7.1 Per-Trial Reconstruction Error (BCE)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
colors = {0: "#888888", 1: "#2196F3", 2: "#E91E63"}
labels_map = {0: "N (noise)", 1: "RN (repeated)", 2: "RefRN (frozen)"}
for t in [0, 1, 2]:
    mask = trial_types_list == t
    ax.scatter(np.where(mask)[0], trial_mean_losses[mask],
               c=colors[t], s=18, alpha=0.7, label=labels_map[t], zorder=3)

# Rolling mean (window=10)
for t in [0, 1, 2]:
    mask = trial_types_list == t
    idx = np.where(mask)[0]
    vals = trial_mean_losses[mask]
    if len(vals) >= 10:
        kernel = np.ones(10) / 10
        smooth = np.convolve(vals, kernel, mode="valid")
        ax.plot(idx[4:4+len(smooth)], smooth, c=colors[t], lw=2, alpha=0.9)

ax.set_xlabel("Trial index")
ax.set_ylabel("Mean BCE per trial")
ax.set_title("CorticalVAE — Online Learning: Per-Trial Reconstruction Error")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.2 ERP-like Response at Repetition Onset

In [ ]:
# Gather BCE around repetition onset for RefRN, RN, and N trials
half_window = 100  # ±100 steps around onset

def get_erp(trial_indices, onset_frame_within_trial=500):
    """Get BCE traces around repetition onset for a set of trials."""
    traces = []
    for tidx in trial_indices:
        trial_start_step = tidx * windows_per_trial
        # The onset_frame_within_trial corresponds to step offset
        center_step = onset_frame_within_trial
        start = max(0, center_step - half_window)
        end = min(windows_per_trial, center_step + half_window)
        trial_losses_slice = all_losses[trial_start_step + start : trial_start_step + end]
        if len(trial_losses_slice) == 2 * half_window:
            traces.append(trial_losses_slice)
    return np.array(traces) if traces else np.empty((0, 2 * half_window))

erp_refrn = get_erp(refrn_indices, 500)
erp_rn = get_erp(rn_indices, 500)
erp_n = get_erp(np.where(trial_order == 0)[0], 500)

fig, ax = plt.subplots(figsize=(10, 5))
t_axis = np.arange(-half_window, half_window)

for data, color, label in [(erp_n, "#888888", "N"),
                             (erp_rn, "#2196F3", "RN"),
                             (erp_refrn, "#E91E63", "RefRN")]:
    if len(data) > 0:
        mean = data.mean(axis=0)
        sem = data.std(axis=0) / np.sqrt(len(data))
        ax.plot(t_axis, mean, c=color, lw=2, label=label)
        ax.fill_between(t_axis, mean - sem, mean + sem, color=color, alpha=0.15)

ax.axvline(0, color="black", ls="--", lw=1, alpha=0.5, label="Repetition onset")
ax.set_xlabel("Frames relative to repetition onset (ms)")
ax.set_ylabel("BCE (reconstruction error)")
ax.set_title("ERP-like Response: BCE Around Repetition Onset")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.3 Distribution Separation

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(10, 5))

for t, color, label in [(0, "#888888", "N"), (1, "#2196F3", "RN"), (2, "#E91E63", "RefRN")]:
    vals = trial_mean_losses[trial_types_list == t]
    if len(vals) > 3:
        kde = gaussian_kde(vals, bw_method=0.3)
        x_range = np.linspace(vals.min() - 10, vals.max() + 10, 300)
        ax.fill_between(x_range, kde(x_range), alpha=0.3, color=color, label=label)
        ax.plot(x_range, kde(x_range), color=color, lw=2)

ax.set_xlabel("Mean BCE per trial")
ax.set_ylabel("Density")
ax.set_title("KDE of Per-Trial BCE by Condition")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 7.4 Rolling d' (Sensitivity Index)

In [ ]:
def rolling_dprime(losses, types, window=20):
    """Compute rolling d' between RefRN and N trials."""
    dprime_vals = []
    dprime_idx = []
    for i in range(window, len(losses)):
        w_losses = losses[i-window:i]
        w_types = types[i-window:i]
        refrn_l = w_losses[w_types == 2]
        n_l = w_losses[w_types == 0]
        if len(refrn_l) >= 3 and len(n_l) >= 3:
            mu_diff = n_l.mean() - refrn_l.mean()
            pooled_std = np.sqrt((n_l.var() + refrn_l.var()) / 2 + 1e-12)
            dprime_vals.append(mu_diff / pooled_std)
            dprime_idx.append(i)
    return np.array(dprime_idx), np.array(dprime_vals)

idx, dp = rolling_dprime(trial_mean_losses, trial_types_list, window=30)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(idx, dp, c="#E91E63", lw=2)
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_xlabel("Trial index")
ax.set_ylabel("d' (N vs RefRN)")
ax.set_title("Rolling d' — Sensitivity to Frozen Noise (RefRN) Over Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Report final d'
if len(dp) > 0:
    print(f"Final d' (last 30 trials): {dp[-1]:.3f}")

### 7.5 Learned STRF Kernels

In [ ]:
# Visualize the learned STRF kernels vs initialization
learned_kernels = model.strf_conv.weight.detach().cpu().numpy()[:, 0, :, :]  # (40, 21, 21)
init_kernels = strf_init.numpy()[:, 0, :, :]  # (40, 21, 21)

n_show = min(10, cfg.n_strf)
fig, axes = plt.subplots(2, n_show, figsize=(20, 4))
vmax = max(abs(init_kernels[:n_show]).max(), abs(learned_kernels[:n_show]).max())

for i in range(n_show):
    axes[0, i].imshow(init_kernels[i], cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    axes[0, i].set_title(f"Init {i}", fontsize=8)
    axes[0, i].axis("off")
    axes[1, i].imshow(learned_kernels[i], cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
    axes[1, i].set_title(f"Learned {i}", fontsize=8)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Initial", fontsize=10)
axes[1, 0].set_ylabel("Learned", fontsize=10)
fig.suptitle("STRF Kernels: Initialization vs Learned (first 10)", fontsize=12)
plt.tight_layout()
plt.show()

# Quantify change
delta = np.linalg.norm(learned_kernels - init_kernels) / np.linalg.norm(init_kernels)
print(f"Relative STRF change (Frobenius): {delta:.4f}")

### 7.6 Latent Space at Repetition Onsets

In [ ]:
from sklearn.decomposition import PCA

# Gather latent vectors
latent_refrn = np.array(latent_at_refrn_rep) if latent_at_refrn_rep else np.empty((0, cfg.latent_dim))
latent_rn = np.array(latent_at_rn_rep) if latent_at_rn_rep else np.empty((0, cfg.latent_dim))

if latent_refrn.shape[0] > 2 and latent_rn.shape[0] > 2:
    all_latent = np.vstack([latent_refrn, latent_rn])
    pca = PCA(n_components=2)
    proj = pca.fit_transform(all_latent)
    n_ref = len(latent_refrn)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(proj[n_ref:, 0], proj[n_ref:, 1], c="#2196F3", s=30, alpha=0.6, label="RN")
    ax.scatter(proj[:n_ref, 0], proj[:n_ref, 1], c="#E91E63", s=30, alpha=0.6, label="RefRN")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    ax.set_title("Latent Space at Repetition Onsets (PCA)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough latent samples for PCA visualization.")

## 8. Discussion**Architecture**: The CorticalVAE uses a bio-inspired STRF frontend initialized fromauditory neuroscience (Chi, Ru & Shamma, 2005), made differentiable followingthe diffAudNeuro approach (Lostanlen et al.). The 40 STRF kernels (5 scales × 8 rates)capture spectro-temporal modulations that the auditory cortex is known to encode.**Online learning**: The model processes one 64ms window at a time with single-sample SGD.This means the model must learn representations *as it encounters stimuli* — there is noreplay buffer or batching. This is a stringent test of whether the architecture canrapidly form memories from noise.**White noise limitation**: Unlike natural sounds, white noise has flat spectro-temporalstatistics. The STRF frontend helps by imposing biologically-plausible spectro-temporalstructure onto the representation, even when the input itself lacks it. The learnableSTRF parameters can adapt to extract the subtle regularities present in repeated segments.**Expected results**:- **Per-trial BCE** should decrease over time as the model learns general noise statistics- **RefRN** trials should show progressively lower BCE as the frozen segment becomes familiar- **ERP at repetition onset**: A prediction-error drop should appear at the 500ms mark  for RN/RefRN but not for N trials- **Rolling d'**: Sensitivity to RefRN should increase over the course of the experiment,  mirroring the behavioral learning curve in Agus et al. (2010)- **STRF kernels**: The learned kernels should deviate from initialization, reflecting  task-driven adaptation of spectro-temporal processing